# OlmOCR benchmark — platform-adaptive

Run [Allen AI's OlmOCR-2](https://github.com/allenai/olmocr)
(Qwen2.5-VL-7B fine-tune) against the
[`pdf-plaintext-extraction`](https://github.com/EvanOchsner/pdf-plaintext-extraction)
synthetic gold-set corpus and emit rows in the standard benchmark
schema (`extractor: "olmocr"`).

**This notebook detects its accelerator and picks a backend:**

| Platform | Backend | Notes |
|---|---|---|
| NVIDIA CUDA GPU | vLLM offline `LLM` | Kaggle T4, Colab, any CUDA host. Best throughput. |
| Apple Silicon (M-series) | MLX via `mlx-vlm` | Runs on the Mac's own GPU (Metal). The validated path. |
| CPU only | — | Stops with a message; a 7B VLM on CPU is impractical. |

Both GPU backends write identical dolma-doc JSONL into a workspace
directory; the shared importer scores it against ground truth.

**Why two backends.** OlmOCR's upstream path is vLLM + CUDA. On a Mac
that is not available at all (vLLM is CUDA-only), so the Apple-Silicon
path goes through MLX instead — same model, same prompt, same output
schema. The CUDA path is the canonical fast route on a real GPU host.

Set `SOURCE_LIMIT` in Phase 0 to subset the corpus (e.g. `10` →
10 sources × 7 variants = 70 PDFs); `None` runs the full 700.

## Phase 0 — Detect platform + configuration

In [ ]:
import pathlib
import platform
import sys

# --- knobs -----------------------------------------------------------
SOURCE_LIMIT = 10        # None = full 700-PDF corpus; N = first N sources x 7 variants
IMAGE_DIM    = 1288      # longest page-image side (px); olmOCR-2 expects ~1288
MAX_TOKENS   = 4096      # generation cap per page
MLX_MODEL    = "mlx-community/olmOCR-2-7B-1025-8bit"   # Apple Silicon
CUDA_MODEL   = "allenai/olmOCR-2-7B-1025"              # CUDA (bf16, loaded fp16 on T4)

# --- accelerator detection ------------------------------------------
def detect_backend() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
    except ImportError:
        pass
    if sys.platform == "darwin" and platform.machine() == "arm64":
        return "mlx"
    return "cpu"

BACKEND = detect_backend()
print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()}")
print(f"BACKEND  : {BACKEND}")
if BACKEND == "cuda":
    import torch
    print(f"GPU      : {torch.cuda.get_device_name(0)}  x{torch.cuda.device_count()}")
elif BACKEND == "mlx":
    print("GPU      : Apple Silicon (Metal) via MLX")
else:
    print("GPU      : none — see Phase 2 (CPU is not a supported run path)")

## Phase 1 — Repo + corpus

Locate the `pdf-plaintext-extraction` repo (use it in place if the
notebook is run from inside a checkout; otherwise clone it), install the
package, and materialize the synthetic corpus. `ensure_corpus()` is
idempotent — it renders the 100 clean PDFs + 6 poisoning techniques
(=700 PDFs) and skips anything already on disk.

In [ ]:
import os
import subprocess


def find_repo() -> pathlib.Path | None:
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        pp = cand / "pyproject.toml"
        if pp.exists() and 'name = "pdf-plaintext-extraction"' in pp.read_text(errors="ignore"):
            return cand
    return None

REPO_DIR = find_repo()
if REPO_DIR is None:
    base = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle").exists() else pathlib.Path.home()
    REPO_DIR = base / "pdf-plaintext-extraction"
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/EvanOchsner/pdf-plaintext-extraction.git", str(REPO_DIR)],
            check=True,
        )
os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], check=False)

# Install the package (core deps: pypdfium2, reportlab, ... — enough for
# corpus materialization + the importer). Backend-specific heavy deps
# are installed in Phase 2.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)

In [ ]:
from pdf_plaintext_extraction.benchmark import ensure_corpus

corpus_root = ensure_corpus()
pdfs = sorted(corpus_root.rglob("*.pdf"))
print("Corpus root:", corpus_root)
print(f"PDFs materialized: {len(pdfs)}  (expect 700)")
assert len(pdfs) == 700, "corpus materialization is incomplete"

# Workspace (dolma docs land here) + final artifact path.
IS_KAGGLE = pathlib.Path("/kaggle").exists()
WORKSPACE = (pathlib.Path("/kaggle/working") if IS_KAGGLE else REPO_DIR / ".tmp") / "olmocr_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = pathlib.Path("/kaggle/working") if IS_KAGGLE else (REPO_DIR / "experiments" / "results")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSONL = ARTIFACT_DIR / "olmocr_rows.jsonl"
print("Workspace:", WORKSPACE)
print("Artifact :", OUT_JSONL)

## Phase 2 — Inference (backend-dispatched)

Each backend installs its own heavy deps, runs OlmOCR over the
(optionally subsetted) corpus, and writes dolma-doc JSONL to
`<workspace>/results/output_all.jsonl`. The two backends are
interchangeable from the importer's point of view.

- **mlx** — delegates to `pdf_plaintext_extraction.benchmark.olmocr_mlx`,
  the validated Apple-Silicon runner (renders pages with pypdfium2,
  runs `mlx-vlm`, frees the Metal buffer cache between pages).
- **cuda** — vLLM offline `LLM.chat` over base64 page images, inline
  here. `dtype="float16"` + a modest `max_model_len` keep it inside a
  16 GB T4; on a bigger card you can raise both.
- **cpu** — stops; a 7B VLM on CPU is not a practical benchmark run.

In [ ]:
import time

WORKSPACE_RESULTS = WORKSPACE / "results"
WORKSPACE_RESULTS.mkdir(parents=True, exist_ok=True)

if BACKEND == "mlx":
    # --- Apple Silicon: delegate to the validated MLX runner ---------
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "mlx-vlm>=0.5"], check=True)
    from pdf_plaintext_extraction.benchmark.olmocr_mlx import run_olmocr_mlx

    t0 = time.perf_counter()
    run_olmocr_mlx(
        corpus_root=corpus_root,
        workspace=WORKSPACE,
        model_name=MLX_MODEL,
        image_dim=IMAGE_DIM,
        max_tokens=MAX_TOKENS,
        source_limit=SOURCE_LIMIT,
        resume=True,
    )
    INFERENCE_WALL = time.perf_counter() - t0
    RUN_MODEL = MLX_MODEL

elif BACKEND == "cuda":
    # --- CUDA: vLLM offline LLM.chat ---------------------------------
    import base64
    import datetime
    import hashlib
    import json
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "vllm==0.11.2", "transformers==4.57.3", "olmocr==0.4.27"],
        check=True,
    )
    # poppler-utils gives pdftoppm, used by render_pdf_pages' fallback.
    subprocess.run(["apt-get", "install", "-y", "-qq", "poppler-utils"], check=False)

    from vllm import LLM, SamplingParams

    # These three helpers are platform-neutral (pure pypdfium2 / string
    # work); they live in olmocr_mlx only because that module exists.
    from pdf_plaintext_extraction.benchmark.olmocr_mlx import (
        OLMOCR_V4_PROMPT,
        render_pdf_pages,
        strip_front_matter,
    )

    # Select PDFs (optionally subset to the first N source_ids).
    run_pdfs = sorted(corpus_root.rglob("*.pdf"))
    if SOURCE_LIMIT is not None:
        keep = set(sorted({p.stem for p in run_pdfs})[:SOURCE_LIMIT])
        run_pdfs = [p for p in run_pdfs if p.stem in keep]

    llm = LLM(
        model=CUDA_MODEL,
        dtype="float16",          # T4 (Turing) has no native bf16 tensor cores
        max_model_len=6144,
        gpu_memory_utilization=0.92,
        enforce_eager=True,
        limit_mm_per_prompt={"image": 1},
    )
    sampling = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)
    render_dir = WORKSPACE / "page_renders"
    out_dolma = WORKSPACE_RESULTS / "output_all.jsonl"
    out_dolma.unlink(missing_ok=True)

    t0 = time.perf_counter()
    with out_dolma.open("a", encoding="utf-8") as fout:
        for i, pdf_path in enumerate(run_pdfs, start=1):
            pdf_t0 = time.perf_counter()
            page_pngs = render_pdf_pages(pdf_path, render_dir, IMAGE_DIM)
            page_texts = []
            for png in page_pngs:
                b64 = base64.b64encode(png.read_bytes()).decode("ascii")
                messages = [{"role": "user", "content": [
                    {"type": "text", "text": OLMOCR_V4_PROMPT},
                    {"type": "image_url",
                     "image_url": {"url": f"data:image/png;base64,{b64}"}},
                ]}]
                out = llm.chat(messages, sampling)
                page_texts.append(strip_front_matter(out[0].outputs[0].text))
            for png in render_dir.glob(f"{pdf_path.stem}_p*.png"):
                png.unlink(missing_ok=True)
            doc_text = "\n".join(t for t in page_texts if t)
            if not doc_text:
                continue
            now = datetime.datetime.now().strftime("%Y-%m-%d")
            fout.write(json.dumps({
                "id": hashlib.sha1(doc_text.encode()).hexdigest(),
                "text": doc_text, "source": "olmocr", "added": now, "created": now,
                "metadata": {
                    "Source-File": str(pdf_path),
                    "olmocr-version": f"vllm:{CUDA_MODEL}",
                    "pdf-total-pages": len(page_texts),
                    "wall_seconds": time.perf_counter() - pdf_t0,
                    "inference": "vllm-offline", "model": CUDA_MODEL,
                    "image_dim": IMAGE_DIM,
                },
            }) + "\n")
            fout.flush()
            if i == 1 or i % 10 == 0 or i == len(run_pdfs):
                el = time.perf_counter() - t0
                print(f"[{i}/{len(run_pdfs)}] {pdf_path.name}  elapsed={el/60:.1f}min")
    INFERENCE_WALL = time.perf_counter() - t0
    RUN_MODEL = CUDA_MODEL

else:
    raise RuntimeError(
        "No supported accelerator. OlmOCR needs a CUDA GPU or Apple "
        "Silicon — a 7B vision LLM on CPU is not a practical run. "
        "On Kaggle: Settings -> Accelerator -> GPU."
    )

print(f"\nInference wall: {INFERENCE_WALL:.1f}s ({INFERENCE_WALL/60:.1f}min)")

## Phase 3 — Score against ground truth → benchmark rows

`import_olmocr_workspace()` walks `<workspace>/results/*.jsonl`, maps
each dolma doc back to its `(source_id, variant)` cell, scores the text
against the ground-truth manifest, and writes rows with
`extractor: "olmocr"` in the standard benchmark schema.

In [ ]:
from pdf_plaintext_extraction.benchmark.olmocr_importer import import_olmocr_workspace

rows = import_olmocr_workspace(
    workspace=WORKSPACE,
    corpus_root=corpus_root,
    out_jsonl=OUT_JSONL,
    strict=False,   # warn-and-continue on cells not covered by a subset run
)
print(f"Wrote {len(rows)} rows -> {OUT_JSONL}")
print(f"  unique cells : {len({(r['source_id'], r['variant']) for r in rows})}")
print(f"  errored rows : {sum(1 for r in rows if r['error'])}")

## Phase 4 — F1 + runtime summary

Two views the project tracks: mean token-F1 per variant (the `clean`
column is the harness control, expect ≥ 0.95) and mean per-PDF wall
time per obfuscation class — `clean` (none), each individual
technique, and the six-technique `poisoned` aggregate.

In [ ]:
from collections import defaultdict

f1_by, wall_by = defaultdict(list), defaultdict(list)
for r in rows:
    if r.get("error"):
        continue
    f1_by[r["variant"]].append(r["f1"])
    if r.get("wall_seconds") is not None:
        wall_by[r["variant"]].append(float(r["wall_seconds"]))

print(f"{'variant':16s}{'n':>5s}{'mean F1':>10s}{'mean wall':>12s}")
print("-" * 43)
for v in sorted(f1_by):
    fs, ws = f1_by[v], wall_by.get(v, [])
    wall = f"{sum(ws)/len(ws):.1f}s" if ws else "n/a"
    print(f"{v:16s}{len(fs):>5d}{sum(fs)/len(fs):>10.3f}{wall:>12s}")

pois_f1  = [f for v, fs in f1_by.items() if v != "clean" for f in fs]
pois_wall = [w for v, ws in wall_by.items() if v != "clean" for w in ws]
if pois_f1:
    wall = f"{sum(pois_wall)/len(pois_wall):.1f}s" if pois_wall else "n/a"
    print(f"\n{'poisoned (all)':16s}{len(pois_f1):>5d}"
          f"{sum(pois_f1)/len(pois_f1):>10.3f}{wall:>12s}")
print(f"\nInference wall (Phase 2): {INFERENCE_WALL/60:.1f}min")

## Phase 5 — Reproducibility marker

In [ ]:
print("=== reproducibility marker ===")
print(f"backend     : {BACKEND}")
print(f"model       : {RUN_MODEL}")
print(f"source limit: {SOURCE_LIMIT}  (None = full 700-PDF corpus)")
print(f"image dim   : {IMAGE_DIM}px longest side")
print(f"max tokens  : {MAX_TOKENS}")
print(f"rows        : {len(rows)}")
print(f"artifact    : {OUT_JSONL}")
print(f"inference   : {INFERENCE_WALL:.1f}s ({INFERENCE_WALL/60:.1f}min)")